# Name Similarity Explorer

This notebook lets you explore the name similarity logic and query specific names.

In [1]:
import sqlite3
import json
import numpy as np
from typing import Optional, List, Dict

# Database path
DB_PATH = "../backend/data/hatch.db"
conn = sqlite3.connect(DB_PATH)
print(f"Connected to {DB_PATH}")

Connected to ../backend/data/hatch.db


## Database Stats

In [2]:
# Quick stats
total_names = conn.execute("SELECT COUNT(*) FROM names").fetchone()[0]
names_with_sims = conn.execute("SELECT COUNT(DISTINCT name_id) FROM name_similarities").fetchone()[0]
total_sims = conn.execute("SELECT COUNT(*) FROM name_similarities").fetchone()[0]
names_with_embeddings = conn.execute(
    "SELECT COUNT(*) FROM name_facts WHERE embedding_phonetic IS NOT NULL"
).fetchone()[0]

print(f"Total names: {total_names:,}")
print(f"Names with pre-computed similarities: {names_with_sims:,}")
print(f"Names WITHOUT pre-computed sims: {total_names - names_with_sims:,}")
print(f"Total similarity pairs: {total_sims:,}")
print(f"Names with embeddings: {names_with_embeddings:,}")

Total names: 17,271
Names with pre-computed similarities: 16,820
Names WITHOUT pre-computed sims: 451
Total similarity pairs: 324,776
Names with embeddings: 16,913


## Method 1: Pre-computed Similarities (Fast)

Looks up pre-computed similarities from the `name_similarities` table.

In [3]:
def find_similar_precomputed(
    name: str,
    top_k: int = 10,
    min_similarity: float = 0.84,
    gender: Optional[str] = None
) -> List[Dict]:
    """Look up pre-computed similar names from database."""
    gender_filter = "AND n2.gender = ?" if gender else ""
    
    query = f"""
        SELECT n2.name, n2.gender, ns.similarity
        FROM name_similarities ns
        JOIN names n1 ON ns.name_id = n1.id
        JOIN names n2 ON ns.similar_name_id = n2.id
        WHERE n1.name = ?
          AND ns.similarity >= ?
          {gender_filter}
        ORDER BY ns.rank
        LIMIT ?
    """
    
    params = [name, min_similarity]
    if gender:
        params.append(gender)
    params.append(top_k)
    
    rows = conn.execute(query, params).fetchall()
    
    return [
        {"name": row[0], "gender": row[1], "similarity": round(row[2], 3)}
        for row in rows
    ]

In [4]:
# Test pre-computed lookup
name = "Emma"
results = find_similar_precomputed(name, top_k=10)
print(f"Similar names to '{name}' (pre-computed):")
for r in results:
    print(f"  {r['name']} ({r['gender']}): {r['similarity']}")

Similar names to 'Emma' (pre-computed):
  Эмма (F): 0.909
  Ema (F): 0.905
  Emmie (F): 0.867
  Emmet (M): 0.86
  Emmeline (F): 0.849
  Emmy (F): 0.844
  Emmeli (F): 0.841


## Method 2: On-the-fly Computation (Slow, loads embeddings)

Computes similarities using phonetic + etymology embeddings. This loads all embeddings into memory.

In [5]:
# Weights for combining embeddings
PHONETIC_WEIGHT = 0.6  # Sound/pronunciation similarity
ETYMOLOGY_WEIGHT = 0.4  # Meaning/origin similarity

# Cache for embeddings
_embeddings_cache = None

def load_embeddings():
    """Load all embeddings into memory (cached)."""
    global _embeddings_cache
    
    if _embeddings_cache is not None:
        return _embeddings_cache
    
    print("Loading embeddings into memory...")
    
    rows = conn.execute("""
        SELECT n.id, n.name, n.gender, nf.embedding_phonetic, nf.embedding_etymology
        FROM names n
        JOIN name_facts nf ON n.id = nf.name_id
        WHERE nf.embedding_phonetic IS NOT NULL OR nf.embedding_etymology IS NOT NULL
    """).fetchall()
    
    phonetic_embeddings = {}
    etymology_embeddings = {}
    name_data = {}
    
    for row in rows:
        name_id, name, gender, phonetic_json, etymology_json = row
        if phonetic_json or etymology_json:
            if phonetic_json:
                phonetic_embeddings[name] = np.array(json.loads(phonetic_json), dtype=np.float32)
            if etymology_json:
                etymology_embeddings[name] = np.array(json.loads(etymology_json), dtype=np.float32)
            name_data[name] = {'id': name_id, 'name': name, 'gender': gender}
    
    print(f"Loaded {len(phonetic_embeddings)} phonetic + {len(etymology_embeddings)} etymology embeddings")
    
    _embeddings_cache = (phonetic_embeddings, etymology_embeddings, name_data)
    return _embeddings_cache


def find_similar_compute(
    name: str,
    top_k: int = 10,
    min_similarity: float = 0.84,
    gender: Optional[str] = None
) -> List[Dict]:
    """Compute similar names on-the-fly using embeddings."""
    phonetic_embeddings, etymology_embeddings, name_data = load_embeddings()
    
    if name not in name_data:
        print(f"Name '{name}' not found in embeddings")
        return []
    
    # Get query embeddings
    query_phonetic = phonetic_embeddings.get(name)
    query_etymology = etymology_embeddings.get(name)
    
    if query_phonetic is None and query_etymology is None:
        print(f"Name '{name}' has no embeddings")
        return []
    
    results = []
    
    for target_name, data in name_data.items():
        if target_name == name:
            continue
        if gender and data['gender'] != gender:
            continue
        
        # Compute weighted similarity
        total_weight = 0.0
        weighted_sim = 0.0
        
        # Phonetic similarity
        if query_phonetic is not None and target_name in phonetic_embeddings:
            target_phonetic = phonetic_embeddings[target_name]
            phonetic_sim = float(np.dot(
                query_phonetic / np.linalg.norm(query_phonetic),
                target_phonetic / np.linalg.norm(target_phonetic)
            ))
            weighted_sim += PHONETIC_WEIGHT * phonetic_sim
            total_weight += PHONETIC_WEIGHT
        
        # Etymology similarity
        if query_etymology is not None and target_name in etymology_embeddings:
            target_etymology = etymology_embeddings[target_name]
            etymology_sim = float(np.dot(
                query_etymology / np.linalg.norm(query_etymology),
                target_etymology / np.linalg.norm(target_etymology)
            ))
            weighted_sim += ETYMOLOGY_WEIGHT * etymology_sim
            total_weight += ETYMOLOGY_WEIGHT
        
        if total_weight == 0:
            continue
        
        final_sim = weighted_sim / total_weight
        
        if final_sim >= min_similarity:
            results.append({
                'name': target_name,
                'gender': data['gender'],
                'similarity': round(final_sim, 3)
            })
    
    # Sort by similarity descending
    results.sort(key=lambda x: x['similarity'], reverse=True)
    return results[:top_k]

In [6]:
# Test on-the-fly computation
name = "Emma"
results = find_similar_compute(name, top_k=10)
print(f"Similar names to '{name}' (computed):")
for r in results:
    print(f"  {r['name']} ({r['gender']}): {r['similarity']}")

Loading embeddings into memory...
Loaded 16603 phonetic + 16603 etymology embeddings
Similar names to 'Emma' (computed):
  Эмма (F): 0.909
  Ema (F): 0.905
  Emmie (F): 0.867
  Emmet (M): 0.86
  Emmeline (F): 0.849
  Emmy (F): 0.844
  Emmeli (F): 0.841


## Query Any Name

Change the `QUERY_NAME` variable to explore different names.

In [9]:
# === CHANGE THIS TO QUERY DIFFERENT NAMES ===
QUERY_NAME = "Olivia"
TOP_K = 15
MIN_SIMILARITY = 0.80
GENDER_FILTER = None  # Set to 'M', 'F', or 'U' to filter
# ============================================

print(f"=" * 50)
print(f"Querying: {QUERY_NAME}")
print(f"=" * 50)

# Check if name exists
name_info = conn.execute(
    "SELECT id, name, gender FROM names WHERE name = ?", 
    (QUERY_NAME,)
).fetchone()

if not name_info:
    print(f"Name '{QUERY_NAME}' not found in database!")
else:
    print(f"Name: {name_info[1]} (gender: {name_info[2]}, id: {name_info[0]})")
    
    # Check for pre-computed similarities
    precomputed = find_similar_precomputed(QUERY_NAME, TOP_K, MIN_SIMILARITY, GENDER_FILTER)
    
    print(f"\nPre-computed similarities: {len(precomputed)} found")
    if precomputed:
        for r in precomputed:
            print(f"  {r['name']:20} ({r['gender']}): {r['similarity']:.3f}")
    else:
        print("  (none - will fall back to on-the-fly computation in production)")
        
        # Try on-the-fly
        print(f"\nComputing on-the-fly...")
        computed = find_similar_compute(QUERY_NAME, TOP_K, MIN_SIMILARITY, GENDER_FILTER)
        if computed:
            for r in computed:
                print(f"  {r['name']:20} ({r['gender']}): {r['similarity']:.3f}")
        else:
            print("  (no similar names found)")

Querying: Olivia
Name: Olivia (gender: F, id: 4d4b1661-d9cf-413f-90fa-2a225f43b75b)

Pre-computed similarities: 12 found
  Olívia               (F): 0.959
  Oliva                (F): 0.880
  Olive                (F): 0.878
  Alivia               (F): 0.834
  Olivia-Rose          (F): 0.829
  Liv                  (F): 0.828
  Livia                (F): 0.824
  Oliver               (M): 0.819
  Oliviero             (M): 0.815
  Olivio               (M): 0.814
  Olivo                (M): 0.814
  Liva                 (F): 0.806


## Find Names Missing Pre-computed Similarities

In [8]:
# Find popular names WITHOUT pre-computed similarities
missing = conn.execute("""
    SELECT n.name, n.gender, SUM(np.weighted_count) as total_weight
    FROM names n
    JOIN name_popularity np ON n.id = np.name_id
    WHERE n.id NOT IN (SELECT DISTINCT name_id FROM name_similarities)
    GROUP BY n.id
    ORDER BY total_weight DESC
    LIMIT 20
""").fetchall()

print("Top 20 popular names WITHOUT pre-computed similarities:")
print(f"{'Name':<20} {'Gender':<8} {'Popularity Weight'}")
print("-" * 50)
for row in missing:
    print(f"{row[0]:<20} {row[1] or 'U':<8} {row[2]:,.0f}")

Top 20 popular names WITHOUT pre-computed similarities:
Name                 Gender   Popularity Weight
--------------------------------------------------
Thomas               M        535,380
Isabel               F        337,555
Jose Luis            M        304,629
Eric                 M        292,258
Francisco Javier     M        291,152
Maria Jose           F        208,498
Giuseppe             M        195,593
Madison              F        190,831
Martin               M        137,430
Ronald               M        125,063
Mohamed              M        113,568
Yolanda              F        113,551
Ignacio              M        99,374
Isaiah               M        96,910
Larry                M        93,579
Madeline             F        69,075
Parker               U        52,450
Russell              M        51,494
Iker                 M        47,249
Maurice              M        44,889


## Check Why Names Are Missing Similarities

Names without embeddings can't have similarities computed.

In [11]:
# Check if missing names have embeddings
check_name = "Thomas"  # Change this

result = conn.execute("""
    SELECT 
        n.name,
        nf.embedding_phonetic IS NOT NULL as has_phonetic,
        nf.embedding_etymology IS NOT NULL as has_etymology
    FROM names n
    LEFT JOIN name_facts nf ON n.id = nf.name_id
    WHERE n.name = ?
""", (check_name,)).fetchone()

if result:
    print(f"Name: {result[0]}")
    print(f"Has phonetic embedding: {bool(result[1])}")
    print(f"Has etymology embedding: {bool(result[2])}")
    
    if not result[1] and not result[2]:
        print("\n=> This name has NO embeddings, so similarities cannot be computed!")
else:
    print(f"Name '{check_name}' not found")

Name: Thomas
Has phonetic embedding: False
Has etymology embedding: False

=> This name has NO embeddings, so similarities cannot be computed!


## Compute Similarities for Missing Names

Run this to compute and store similarities for names that are missing them.

In [ ]:
def compute_and_store_similarities(
    name: str,
    top_k: int = 20,
    min_similarity: float = 0.84,
    dry_run: bool = True
):
    """Compute similarities for a name and optionally store them."""
    # Get name_id
    result = conn.execute("SELECT id FROM names WHERE name = ?", (name,)).fetchone()
    if not result:
        print(f"Name '{name}' not found")
        return
    
    name_id = result[0]
    
    # Check if already has similarities
    existing = conn.execute(
        "SELECT COUNT(*) FROM name_similarities WHERE name_id = ?",
        (name_id,)
    ).fetchone()[0]
    
    if existing > 0:
        print(f"Name '{name}' already has {existing} pre-computed similarities")
        return
    
    # Compute similarities
    similar = find_similar_compute(name, top_k=top_k, min_similarity=min_similarity)
    
    if not similar:
        print(f"No similar names found for '{name}' (may be missing embeddings)")
        return
    
    print(f"Found {len(similar)} similar names for '{name}':")
    for i, s in enumerate(similar):
        print(f"  {i+1}. {s['name']} ({s['gender']}): {s['similarity']:.3f}")
    
    if dry_run:
        print(f"\n[DRY RUN] Would insert {len(similar)} rows. Set dry_run=False to save.")
        return
    
    # Insert into database
    for rank, s in enumerate(similar, 1):
        similar_id = conn.execute(
            "SELECT id FROM names WHERE name = ?",
            (s['name'],)
        ).fetchone()[0]
        
        conn.execute("""
            INSERT INTO name_similarities (name_id, similar_name_id, similarity, rank)
            VALUES (?, ?, ?, ?)
        """, (name_id, similar_id, s['similarity'], rank))
    
    conn.commit()
    print(f"\nInserted {len(similar)} similarities for '{name}'")

In [ ]:
# Test computing similarities for a missing name
compute_and_store_similarities("Thomas", dry_run=True)

In [ ]:
# Batch compute for all missing popular names (BE CAREFUL - this modifies the DB!)
# Set DRY_RUN = False to actually save

DRY_RUN = True
LIMIT = 10  # How many names to process

missing_names = conn.execute("""
    SELECT n.name
    FROM names n
    JOIN name_popularity np ON n.id = np.name_id
    WHERE n.id NOT IN (SELECT DISTINCT name_id FROM name_similarities)
    GROUP BY n.id
    ORDER BY SUM(np.weighted_count) DESC
    LIMIT ?
""", (LIMIT,)).fetchall()

print(f"Processing {len(missing_names)} names (dry_run={DRY_RUN})...\n")

for (name,) in missing_names:
    print(f"--- {name} ---")
    compute_and_store_similarities(name, dry_run=DRY_RUN)
    print()

In [ ]:
# Close connection when done
# conn.close()